# Task 2 — RL top-N Re-rank (Actor-Critic policy-gradient, KHÔNG TD3+BC)

BERT (Task 1) đóng băng → top-N=100 ứng viên → Policy(Actor)+Value(Critic) sắp lại. Reward = graded NDCG. Policy init ngẫu nhiên. So BERT vs BERT+RL.

In [1]:
import os, sys, json, torch
import torch.optim as optim
# path: chạy được dù cwd ở repo-root hay trong longterm_rerank
HERE = os.path.abspath("longterm_rerank") if os.path.isdir("longterm_rerank") else os.getcwd()
sys.path.insert(0, HERE)
import common, run_pipeline as R
from common import set_seed, load_and_split, BERT4Rec, BERT4RecDataset
set_seed(42)
DEVICE = R.DEVICE
MAX_LEN = R.MAX_LEN
print("Device:", DEVICE, "| HERE:", HERE)
import wandb

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\TanPhat\_netrc


wandb: Currently logged in as: lamgiang (lamgiang-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda | HERE: F:\1_REL\Reinforcement-learning-for-Recommendation\longterm_rerank


## 1. Nạp split (cùng seed) + BERT đã train (Task 1, đóng băng)

In [2]:
users, bert_seqs, n_real, movie2id, stats = load_and_split(R.RATING_FILE, max_len=MAX_LEN)
n_model = n_real + 1
bert = BERT4Rec(n_items=n_model, max_seq_length=MAX_LEN, hidden_size=64,
                n_layers=2, n_heads=2, hidden_dropout_prob=0.2,
                attn_dropout_prob=0.2, loss_type="CE").to(DEVICE)
bert.load_state_dict(torch.load(os.path.join(HERE,"bert4rec_longterm.pth"), map_location=DEVICE))
bert.eval()
for p in bert.parameters(): p.requires_grad = False
print("BERT loaded & frozen")

BERT loaded & frozen


C:\Users\TanPhat\AppData\Local\Temp\ipykernel_15896\331952056.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bert.load_state_dict(torch.load(os.path.join(HERE,"bert4rec

## 2. Train RL re-ranker (Actor-Critic PG). `train_rl`/`PolicyNet` ở run_pipeline.py

In [3]:
key = common.load_env_key(os.path.join(os.path.dirname(HERE), ".env"))
run = None
try:
    if key: wandb.login(key=key)
    run = wandb.init(project="bert4rec-longterm-rerank", name="nb-rl-rerank-ac", reinit=True)
except Exception as e:
    print("wandb off:", e)
policy = R.train_rl(bert, users, DEVICE, epochs=30, bs=512, topN=100, K=10, patience=8, run=run)
torch.save(policy.state_dict(), os.path.join(HERE,"rl_reranker.pth"))
if run:
    try: wandb.finish()
    except Exception: pass
print("saved rl_reranker.pth")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\TanPhat\_netrc


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run 694zzve3


...

wandb: uploading config.yaml; uploading output.log


wandb: 
wandb: Run history:
wandb:        rl/epoch ▁▂▃▄▅▅▆▇█
wandb: rl/train_reward ▅▇▃▅▆█▃▁▆
wandb:   rl/val_ndcg10 ███████▁▅
wandb: 
wandb: Run summary:
wandb:        rl/epoch 9
wandb: rl/train_reward 0.05484
wandb:   rl/val_ndcg10 0.0394
wandb: 


wandb:  View run nb-rl-rerank-ac at: https://wandb.ai/lamgiang-fpt-university/bert4rec-longterm-rerank/runs/694zzve3
wandb:  View project at: https://wandb.ai/lamgiang-fpt-university/bert4rec-longterm-rerank
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260619_183609-694zzve3\logs


saved rl_reranker.pth


## 3. Đánh giá: BERT vs BERT+RL trên TEST (cùng metric, cùng tập)

In [4]:
res_bert = R.eval_bert(bert, users, DEVICE, ks=(5,10,20))
res_rl   = R.eval_rl(bert, policy, users, DEVICE, target="test", ks=(5,10,20), topN=100)
results = {"data_stats": stats, "BERT (retrained)": res_bert, "BERT + RL re-rank": res_rl}
json.dump(results, open(os.path.join(HERE,"results.json"),"w"), ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "data_stats": {
    "n_users": 6040,
    "n_items": 3706,
    "n_interactions": 1000209,
    "n_train_remaining": 800193,
    "n_test": 200016,
    "avg_seq_len": 165.6,
    "avg_test_len": 33.12,
    "test_frac": 0.2,
    "max_len": 200
  },
  "BERT (retrained)": {
    "NDCG@5": 0.0344728411532718,
    "Recall@5": 0.011953588171009019,
    "NDCG@10": 0.03694276721597422,
    "Recall@10": 0.022047979062496392,
    "NDCG@20": 0.048731478363197855,
    "Recall@20": 0.0473534095949075,
    "MeanRating@10": 1.2210292494481236
  },
  "BERT + RL re-rank": {
    "NDCG@5": 0.0344728411532718,
    "Recall@5": 0.011953588171009019,
    "NDCG@10": 0.036943706370209134,
    "Recall@10": 0.022047979062496392,
    "NDCG@20": 0.04873157583400964,
    "Recall@20": 0.0473534095949075,
    "MeanRating@10": 1.2210292494481236
  }
}


## 4. Bảng so sánh

In [5]:
COLS=["NDCG@5","NDCG@10","NDCG@20","Recall@10","MeanRating@10"]
print(f"{'Model':<22}"+"".join(f"{c:>13}" for c in COLS))
print("-"*(22+13*len(COLS)))
for name,m in [("BERT (retrained)",res_bert),("BERT + RL re-rank",res_rl)]:
    print(f"{name:<22}"+"".join(f"{m[c]:>13.4f}" for c in COLS))

Model                        NDCG@5      NDCG@10      NDCG@20    Recall@10MeanRating@10
---------------------------------------------------------------------------------------
BERT (retrained)             0.0345       0.0369       0.0487       0.0220       1.2210
BERT + RL re-rank            0.0345       0.0369       0.0487       0.0220       1.2210
